# 📋 Procesador de Planificaciones DEL — UST 2026-1

---

## Qué necesitas

| | Archivo | Obligatorio |
|-|---------|-------------|
| 📄 | PDF del programa de asignatura | ✅ Sí |
| 📊 | Planificación didáctica `.xlsx` | ✅ Sí |
| 📌 | Activar `ES_AS = True` | Solo si la asignatura tiene lineamiento A+S |
| 📜 | Activar `DECRETO_HABILITADO = True` | Solo si existe un decreto que actualice el programa |

---

## Pasos

| Celda | Qué hace |
|-------|----------|
| **Celda 1** | Instala dependencias — ejecutar una vez al abrir la sesión |
| **Celda 2** | Sube el **PDF del programa** y configura las opciones si aplican |
| **Celda 3** | Sube el **`.xlsx`**, lo corrige y descarga el resultado |

---

## Resultado

El archivo descargado incluye las correcciones marcadas en **azul** y un log con:
- Correcciones aplicadas fila por fila
- Estado de los 41 criterios de la escala UST
- Discrepancias respecto al programa oficial
- *(si A+S)* Estado de los 11 hitos del lineamiento A+Se UST 2025

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELDA 1 — Instalar dependencias (una vez por sesión)               ║
# ╚══════════════════════════════════════════════════════════════════════╝
import subprocess, sys

for pkg in ['openpyxl', 'pdfplumber']:
    print(f'📦 Instalando {pkg}...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
    print(f'✅ {pkg} listo')

print('\n✅ Dependencias instaladas. Ejecuta la Celda 2.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELDA 2 — Subir PDF del programa y extraer datos                   ║
# ╚══════════════════════════════════════════════════════════════════════╝
import base64, gzip, os, sys, importlib
from google.colab import files

# ── Script embebido ───────────────────────────────────────────────────────
_SCRIPT_B64 = """
...
JgEA
"""

script_bytes = gzip.decompress(base64.b64decode(_SCRIPT_B64.strip()))
with open('/content/revisar_planificaciones.py', 'wb') as _f:
    _f.write(script_bytes)
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
import revisar_planificaciones as rp
importlib.reload(rp)

# ── Subir PDF ─────────────────────────────────────────────────────────────
PROGRAMA = {}

print('📄 Sube el programa de asignatura (.pdf)...')
subidos = files.upload()

for nombre, contenido in subidos.items():
    if not nombre.lower().endswith('.pdf'):
        print(f'⚠️  {nombre} no es .pdf — omitido')
        continue

    ruta_pdf = f'/content/{nombre}'
    with open(ruta_pdf, 'wb') as f:
        f.write(contenido)

    PROGRAMA = rp.extraer_programa_pdf(ruta_pdf)

    if PROGRAMA.get('_error'):
        print(f'❌ Error al extraer el PDF: {PROGRAMA["_error"]}')
    else:
        print(f'\n✅ Programa extraído: {nombre}')
        print(f'   Código      : {PROGRAMA.get("codigo", "—")}')
        print(f'   Asignatura  : {PROGRAMA.get("asignatura", "—")}')
        print(f'   Créditos    : {PROGRAMA.get("creditos", "—")}')
        print(f'   Área        : {PROGRAMA.get("area", "—")}')
        print(f'   Horas TPE   : {PROGRAMA.get("horas_tpe", "—")}')
        unidades = PROGRAMA.get('unidades', [])
        print(f'   Unidades    : {len(unidades)}')
        for u in unidades:
            print(f'     • Unidad {u["numero"]}: {u["nombre"]} ({u["horas"]}h)')
        ponds = PROGRAMA.get('ponderaciones', {})
        if ponds:
            print(f'   Ponderaciones: {", ".join(f"U{k}={v}%" for k,v in ponds.items())}')
        if PROGRAMA.get('pct_examen'):
            print(f'   Examen LGA  : {PROGRAMA["pct_examen"]}%')

print('\n✅ Ejecuta la Celda 3 para procesar la planificación.')


# ╔══════════════════════════════════════════════════════════════════════╗
# ║  OPCIONES — modifica solo si corresponde, deja el resto como está   ║
# ╚══════════════════════════════════════════════════════════════════════╝

# ── Opción 1: Lineamiento A+S ─────────────────────────────────────────────
# ¿La asignatura tiene lineamiento Aprendizaje + Servicio (A+Se)?
#   True  → el script verifica los 11 hitos obligatorios del lineamiento UST
#   False → se omite esta verificación (valor por defecto)
ES_AS = False

# ── Opción 2: Decreto de actualización (pendiente) ───────────────────────
# ¿Existe un decreto que actualice el programa vigente?
#   Cuando esté disponible un ejemplo real, se habilitará esta opción.
#   Por ahora déjalo en False.
DECRETO_HABILITADO = False

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELDA 3 — Subir .xlsx → Procesar → Descargar _REVISADO.xlsx        ║
# ╚══════════════════════════════════════════════════════════════════════╝
import glob, shutil
from google.colab import files

# PROGRAMA viene de la Celda 2 (si no se ejecutó, queda vacío)
_programa = PROGRAMA if 'PROGRAMA' in dir() else {}
_es_as    = ES_AS    if 'ES_AS'    in dir() else False

print('📤 Selecciona la planificación .xlsx...')
subidos = files.upload()

for nombre_xlsx, contenido in subidos.items():
    if not nombre_xlsx.endswith('.xlsx'):
        print(f'⚠️  {nombre_xlsx} no es .xlsx — omitido')
        continue

    print(f'\n📂 Procesando: {nombre_xlsx}')

    base_nombre  = nombre_xlsx.replace('.xlsx', '')
    carpeta_asig = f'/tmp/DEL/{base_nombre}'
    carpeta_env  = os.path.join(carpeta_asig, 'Enviado a DEL')
    carpeta_rev  = os.path.join(carpeta_asig, 'Revisado')

    if os.path.exists(carpeta_asig):
        shutil.rmtree(carpeta_asig)
    os.makedirs(carpeta_env, exist_ok=True)

    ruta_xlsx = os.path.join(carpeta_env, nombre_xlsx)
    with open(ruta_xlsx, 'wb') as f:
        f.write(contenido)

    log, ok = rp.procesar_asignatura(carpeta_asig, programa=_programa, es_as=_es_as)
    print('\n'.join(log))

    if not ok:
        print('\n❌ El procesamiento falló — revisa el log.')
        continue

    resultados = glob.glob(os.path.join(carpeta_rev, '*.xlsx'))
    if not resultados:
        print('\n❌ No se generó archivo de salida en Revisado/')
        continue

    ruta_salida   = resultados[0]
    nombre_salida = os.path.basename(ruta_salida)

    print(f'\n💾 Descargando: {nombre_salida}...')
    files.download(ruta_salida)
    print('✅ Descarga iniciada.')

    shutil.rmtree(carpeta_asig, ignore_errors=True)